# CS3255 NLP Lab Assignment 4: Applied POS Tagging & NER

## Aim
To apply POS Tagging, Named Entity Recognition (NER), and performance metrics evaluation to practical NLP tasks, including sentiment feature extraction, GDPR data redaction, and model auditing.

## Required Libraries and Setup
The following cell installs and imports the required libraries for this lab, including `nltk`, `spacy`, and `scikit-learn`, and downloads the English model for `spacy`.

In [2]:
import nltk
import spacy
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

nlp = spacy.load("en_core_web_sm")

## Task 1 – Sentiment Feature Extractor (POS Tagging)

**Scenario:**
ShopSmart wants to identify product features and the adjectives describing those features in customer reviews.

**Linguistic logic:**
- Nouns usually represent product features.
- Adjectives represent opinions/sentiment.
- spaCy POS tagging and dependency relationships can be used to identify adjective–noun relationships.

In [3]:
text_1 = "The new phone has a fantastic camera but the battery life is terrible. The screen is bright , but the price is high ."
doc_1 = nlp(text_1)

print(f"{'Token':<15} | {'POS':<10} | {'Dependency':<15}")
print("-" * 45)
for token in doc_1:
    print(f"{token.text:<15} | {token.pos_:<10} | {token.dep_:<15}")

print("\nFeature Pairs Detected:")

feature_pairs = []

for token in doc_1:
    if token.pos_ in ["NOUN", "PROPN"]:
        for child in token.children:
            if child.pos_ == "ADJ" and child.dep_ == "amod":
                if child.text.lower() != "new":
                    feature_pairs.append((child.text, token.text))
        
        if token.dep_ in ["nsubj", "nsubjpass"]:
            head_verb = token.head
            if head_verb.pos_ in ["AUX", "VERB"]:
                for child in head_verb.children:
                    if child.pos_ == "ADJ" and child.dep_ in ["acomp", "attr"]:
                        feature_pairs.append((child.text, token.text))

unique_pairs = []
for pair in feature_pairs:
    if pair not in unique_pairs:
        unique_pairs.append(pair)

for adj, noun in unique_pairs:
    print(f"- ({adj}, {noun})")

Token           | POS        | Dependency     
---------------------------------------------
The             | DET        | det            
new             | ADJ        | amod           
phone           | NOUN       | nsubj          
has             | VERB       | ROOT           
a               | DET        | det            
fantastic       | ADJ        | amod           
camera          | NOUN       | dobj           
but             | CCONJ      | cc             
the             | DET        | det            
battery         | NOUN       | compound       
life            | NOUN       | nsubj          
is              | AUX        | conj           
terrible        | ADJ        | acomp          
.               | PUNCT      | punct          
The             | DET        | det            
screen          | NOUN       | nsubj          
is              | AUX        | ROOT           
bright          | ADJ        | acomp          
,               | PUNCT      | punct          
but           

**Explanation:**
- Nouns act as the core entity or product feature being discussed (e.g., 'camera', 'life', 'screen', 'price').
- Adjectives act as the descriptive sentiment words attached to those features (e.g., 'fantastic', 'terrible', 'bright', 'high').
- Simple positional proximity (looking for adjacent words) is not sufficient because English syntax often separates adjectives from nouns, especially with linking verbs (e.g., "The screen is bright"). Dependency relationships allow us to traverse the sentence structure and accurately pair the sentiment (adjective) with its target feature (noun).

## Task 2 – The GDPR Redactor (NER)

Named Entity Recognition (NER) identifies entities such as `PERSON` (people, including fictional) and `GPE` (Geopolitical entities, i.e., countries, cities, states).
For this task, we will locate these specific entity types within text and redact them for privacy reasons.

In [4]:
text_2 = "Startups in Bangalore are growing fast. Mr. Rajesh Kumar met with Sarah Jones in London to discuss the merger ."

def redact_sensitive_info(text):
    doc = nlp(text)
    
    print(f"{'Entity':<20} {'Label':<10}")
    print("-" * 30)
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "GPE"]:
            print(f"{ent.text:<20} {ent.label_:<10}")
        
    print("\nRedacted Text:")
    
    result = []
    for token in doc:
        if token.ent_iob_ != "": 
            if token.ent_type_ in ["PERSON", "GPE"]:
                if token.ent_iob_ == "B": 
                    result.append("[REDACTED]" + token.whitespace_)
                elif token.ent_iob_ == "I": 
                    result[-1] = "[REDACTED]" + token.whitespace_ 
            else:
                result.append(token.text + token.whitespace_)
        else:
            result.append(token.text + token.whitespace_)
            
    return "".join(result)

print(redact_sensitive_info(text_2))

Entity               Label     
------------------------------
Bangalore            PERSON    
Rajesh Kumar         PERSON    
Sarah Jones          PERSON    
London               GPE       

Redacted Text:
Startups in [REDACTED] are growing fast. Mr. [REDACTED] met with [REDACTED] in [REDACTED] to discuss the merger .


**Explanation:**
- **NER (Named Entity Recognition)** extracts and categorizes specific entities from text (like names, organizations, or locations).
- **PERSON** refers to people's names (e.g., Rajesh Kumar, Sarah Jones).
- **GPE** stands for Geopolitical Entities, referring to countries, cities, or states (e.g., Bangalore, London).
- Automatic redaction using NER is highly useful for automated privacy compliance (like GDPR or HIPAA), ensuring sensitive information is removed consistently at scale without manual intervention.

## Task 3 – Model Auditing (Metrics)

In [5]:
y_true = ["O", "DRUG", "O", "DRUG", "O", "O"]
y_pred = ["O", "O", "O", "DRUG", "O", "DRUG"]

tp = sum(1 for true, pred in zip(y_true, y_pred) if true == "DRUG" and pred == "DRUG")
fp = sum(1 for true, pred in zip(y_true, y_pred) if true != "DRUG" and pred == "DRUG")
fn = sum(1 for true, pred in zip(y_true, y_pred) if true == "DRUG" and pred != "DRUG")

print(f"TP = {tp}")
print(f"FP = {fp}")
print(f"FN = {fn}\n")

precision = precision_score(y_true, y_pred, pos_label="DRUG", zero_division=0)
recall = recall_score(y_true, y_pred, pos_label="DRUG", zero_division=0)
f1 = f1_score(y_true, y_pred, pos_label="DRUG", zero_division=0)

print("DRUG Metrics")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}\n")

print("Classification Report:")
print(classification_report(y_true, y_pred, zero_division=0))

TP = 1
FP = 1
FN = 1

DRUG Metrics
Precision: 0.50
Recall: 0.50
F1-Score: 0.50

Classification Report:
              precision    recall  f1-score   support

        DRUG       0.50      0.50      0.50         2
           O       0.75      0.75      0.75         4

    accuracy                           0.67         6
   macro avg       0.62      0.62      0.62         6
weighted avg       0.67      0.67      0.67         6



✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


**Analysis:**
- **False Positive (FP):** The model labels a token as `DRUG` when it is not actually a drug.
- **False Negative (FN):** The model fails to identify an actual `DRUG` token.
- In this specific example, **FP = 1** and **FN = 1**. Therefore, neither error type is higher based strictly on the provided example. It is inaccurate to claim the model has "high false negatives" based only on this test set.
- However, in a medical or hospital setting, false negatives can be particularly concerning because a missed medication name could cause critical information to be overlooked by doctors. 
- Ultimately, this tiny six-token example is not sufficient to conclude the overall performance of the deployed chatbot. A much larger and more representative evaluation dataset would be required for meaningful model auditing.

## Final Analysis and Conclusion

1. **spaCy POS tagging** and dependency parsing were successfully used to identify semantic adjective-noun feature relationships.
2. **spaCy NER** was applied to accurately identify `PERSON` and `GPE` entities and redact them without hard-coding rules.
3. **scikit-learn** was used to systematically calculate Precision, Recall, and F1 for the `DRUG` class.
4. The metrics obtained for the `DRUG` class were **Precision = 0.50**, **Recall = 0.50**, and **F1 = 0.50**.
5. The supplied example had an equal number of FP and FN counts (1 each). While false negatives can have severe consequences in a medical medication extraction context, we cannot conclude the model generally suffers from high false negatives from this single sample.
6. A significantly larger dataset is required for comprehensive and meaningful model auditing in production.